# 02_quality_audit.ipynb: How clean is the data? (Week 2 quality audit)

**Course:** DSA 1080A, Programming for Data Science (Future Data Lab group project)
**Checkpoint:** Week 2, item 4 (check null counts and percentages, duplicate rows, invalid numeric ranges, inconsistent categories and date parsing; do not silently change the source data)

This notebook audits the two raw survey files (2023-24 and 2024-25) and ends with a **quality audit table**: one row per check and survey year, with the evidence (a count), a severity and an idea for how Week 3 could treat it.

*AI assistance note: this notebook was drafted with help from Claude (an AI assistant). The author has run it and can explain every cell.*

## Rules of this audit

1. **Flag, don't fix.** Nothing in this notebook changes the raw files or the loaded tables. Cleaning decisions belong to Week 3.
2. **Each survey year is audited separately,** then the two are compared.
3. **Not every blank is an error.** People who do not use ChatGPT skip most questions, and some people stop part-way through. The audit tries to tell these apart from real problems.
4. **Thresholds are provisional.** They are all in one settings cell (Section 0) so they are easy to change once the data dictionary and questionnaires give the official valid ranges.
5. **Severity scale:** *High* = could change a result or breaks a project rule; *Medium* = needs a documented decision; *Low* = minor or expected; *OK* = checked and nothing wrong found.

## 0. Setup: load the data and set the thresholds

In [1]:
import pandas as pd

raw_folder = "../data/raw/"
file_names = {"2023-24": "raw_2023-2024.csv", "2024-25": "raw_2024-2025.csv"}
survey_years = list(file_names.keys())

dataframes = {}                                   # survey year -> table
for year in survey_years:
    dataframes[year] = pd.read_csv(raw_folder + file_names[year], low_memory=False)
    rows, columns = dataframes[year].shape
    print(f"Loaded {year}: {rows:,} rows x {columns} columns")

# ---------- Settings: all thresholds in one place (PROVISIONAL) ----------
near_empty_cutoff = 20     # a row with this many answered columns or fewer counts as 'near-empty'
age_min = 18               # lowest plausible age: nobody in the 2024-25 data is under 18 (replace with the dictionary's range)
age_max = 90               # highest plausible age for a student (our own generous upper bound)
birth_year_min = 1900      # numbers from here up look like birth years typed instead of ages
alert_null_pct = 20        # a column more than this % blank is 'sparse'
usable_warn_pct = 85       # fewer usable rows than this % = Medium concern (our own rule)
usable_alert_pct = 60      # fewer usable rows than this % = High concern (our own rule)

# ---------- The audit table is built row by row in this list ----------
audit_rows = []

Loaded 2023-24: 23,218 rows x 174 columns


Loaded 2024-25: 22,963 rows x 180 columns


## 1. Missing values (nulls)

We use four bands for a column: **Complete** (under 1% blank), **Mostly complete** (1-5%), **Incomplete** (5-20%) and **Sparse** (over 20%).

In [2]:
for year, df in dataframes.items():
    blank_pct = df.isna().mean() * 100            # one number per column

    n_complete = 0
    n_mostly = 0
    n_incomplete = 0
    n_sparse = 0
    for pct in blank_pct:
        if pct < 1:
            n_complete += 1
        elif pct < 5:
            n_mostly += 1
        elif pct <= alert_null_pct:
            n_incomplete += 1
        else:
            n_sparse += 1

    print("=" * 70)
    print(f"{year}: {df.shape[1]} columns")
    print(f"  Complete (<1% blank)      : {n_complete}")
    print(f"  Mostly complete (1-5%)    : {n_mostly}")
    print(f"  Incomplete (5-20%)        : {n_incomplete}")
    print(f"  Sparse (>{alert_null_pct}% blank)        : {n_sparse}")
    print("  The 8 blankest columns (% blank):")
    print(blank_pct.sort_values(ascending=False).head(8).round(1).to_string())

    audit_rows.append({
        "check": "Blank values, whole file",
        "where": "all columns",
        "wave": year,
        "finding": f"{n_sparse} of {df.shape[1]} columns are more than {alert_null_pct}% blank; complete columns (<1% blank): {n_complete}",
        "count": n_sparse,
        "out_of": df.shape[1],
        "severity": "Medium",
        "week3_idea": "Keep the rows. Do not fill or drop for blankness alone. Record as expected missing (see next check).",
    })

2023-24: 174 columns
  Complete (<1% blank)      : 3
  Mostly complete (1-5%)    : 10
  Incomplete (5-20%)        : 1
  Sparse (>20% blank)        : 160
  The 8 blankest columns (% blank):
Q32d    46.4
Q32i    46.4
Q32g    46.4
Q32j    46.4
Q32m    46.4
Q32e    46.4
Q32c    46.4
Q32h    46.4
2024-25: 180 columns
  Complete (<1% blank)      : 1
  Mostly complete (1-5%)    : 11
  Incomplete (5-20%)        : 33
  Sparse (>20% blank)        : 135
  The 8 blankest columns (% blank):
Q13f_text    94.6
Q32m         31.5
Q32g         31.5
Q32n         31.5
Q32j         31.5
Q32h         31.5
Q32l         31.5
Q32k         31.5


### Who is blank? Rows that answered Q15 versus rows that did not

Q15 is the overall-use question, so a blank Q15 means the person is a non-user or stopped early. We compare the blank rate in each question block (Q1, Q2, ... Q40) for the two groups. The usual pattern is:

* **Background questions** (country, age, field of study) are answered by almost everyone.
* **ChatGPT-use questions** are blank for almost all Q15-blank rows. That is *expected* (skip logic), not an error.
* If the blank rate among people who **did** answer Q15 climbs towards the end of the questionnaire, people are **dropping out part-way** (this is a real quality problem).

In [3]:
for year, df in dataframes.items():
    answered_q15 = df["Q15"].notna()
    blank_users = df[answered_q15].isna().mean() * 100          # blank % among rows that answered Q15
    blank_others = df[~answered_q15].isna().mean() * 100        # blank % among rows that did not

    numbers = df.columns.str.extract(r"^Q(\d+)")[0]             # '18' for Q18a, NaN for 'source'
    table = pd.DataFrame({
        "question_no": numbers.values,
        "blank_pct_answered_Q15": blank_users.values,
        "blank_pct_Q15_blank": blank_others.values,
    })
    table = table.dropna(subset=["question_no"])
    table["question_no"] = table["question_no"].astype(int)
    summary = table.groupby("question_no").mean().round(1)       # average over each block's columns

    n_users = int(answered_q15.sum())
    n_others = int((~answered_q15).sum())
    print("=" * 70)
    print(f"{year}: {n_users:,} rows answered Q15, {n_others:,} rows did not ({n_others / len(df) * 100:.1f}%)")
    print("Average blank % per question block:")
    print(summary.to_string())

    audit_rows.append({
        "check": "Blank values: Q15 blank (non-users or stopped early)",
        "where": "Q15 and all ChatGPT-use questions",
        "wave": year,
        "finding": f"{n_others:,} rows have Q15 blank; the ChatGPT-use questions are blank for nearly all of them",
        "count": n_others,
        "out_of": len(df),
        "severity": "Low",
        "week3_idea": "Expected (skip logic). Keep all rows; filter to users inside each analysis; mark as 'expected missing' in the data dictionary.",
    })

2023-24: 15,927 rows answered Q15, 7,291 rows did not (31.4%)
Average blank % per question block:
             blank_pct_answered_Q15  blank_pct_Q15_blank
question_no                                             
1                               5.2                  9.1
2                               0.3                  1.6
3                               0.3                  2.2
4                               0.6                  2.9
5                               1.5                  5.5
6                               1.2                  4.7
7                               0.9                  4.3
8                               1.0                  4.8
9                               1.1                  4.9
10                              1.2                  5.6
11                              0.8                  4.7
12                              0.7                  4.7
13                              0.0                 11.0
14                              0.2            

### Do people who started the ChatGPT questions finish them?

Here we look at **only the rows that answered Q15** and check the blank rate for the questions our objectives rely on. If these are far above 0%, the usable number of rows is smaller than the headline row count.

In [4]:
key_columns = ["Q18a", "Q22a", "Q24a", "Q26a", "Q27a", "Q32a", "Q33", "Q35a"]

for year, df in dataframes.items():
    users = df[df["Q15"].notna()]
    print("=" * 70)
    print(f"{year}: blank % among the {len(users):,} rows that answered Q15")

    rates = {}
    for name in key_columns:
        rates[name] = users[name].isna().mean() * 100
        print(f"  {name:5} {rates[name]:5.1f}%")

    low = min(rates.values())
    high = max(rates.values())
    worst = max(rates, key=rates.get)               # the column with the highest blank rate
    audit_rows.append({
        "check": "Blank values: drop-off among ChatGPT users",
        "where": "Q18a, Q22a, Q24a, Q26a, Q27a, Q32a, Q33, Q35a",
        "wave": year,
        "finding": f"Among rows that answered Q15, {low:.1f}%-{high:.1f}% are blank on the key objective questions (highest: {worst})",
        "count": int(users[worst].isna().sum()),
        "out_of": len(users),
        "severity": "High",
        "week3_idea": "Do not drop these rows from the cleaned file. For each analysis keep only rows with answers in the columns it uses, and report that n.",
    })

2023-24: blank % among the 15,927 rows that answered Q15
  Q18a    0.2%
  Q22a    9.3%
  Q24a   12.7%
  Q26a   16.0%
  Q27a   16.0%
  Q32a   21.8%
  Q33    23.7%
  Q35a   23.6%


2024-25: blank % among the 19,588 rows that answered Q15
  Q18a    0.4%
  Q22a    8.4%
  Q24a   11.5%
  Q26a   14.3%
  Q27a   14.3%
  Q32a   19.7%
  Q33    21.4%
  Q35a   21.4%


### Near-empty rows

Note: Rows with very few answers only contain the opening background questions.

In [5]:
for year, df in dataframes.items():
    answered_per_row = df.notna().sum(axis=1)
    near_empty = answered_per_row <= near_empty_cutoff
    n_near_empty = int(near_empty.sum())

    print("=" * 70)
    print(f"{year}: answered columns per row")
    print(answered_per_row.describe().round(1).to_string())
    print(f"Rows with {near_empty_cutoff} or fewer answered columns: {n_near_empty:,} ({n_near_empty / len(df) * 100:.1f}%)")

    audit_rows.append({
        "check": "Near-empty rows",
        "where": "whole row",
        "wave": year,
        "finding": f"{n_near_empty:,} rows have {near_empty_cutoff} or fewer answered columns (of {df.shape[1]})",
        "count": n_near_empty,
        "out_of": len(df),
        "severity": "Low",
        "week3_idea": "Flag with a True/False column (e.g. near_empty). Keep them: they still carry the background answers.",
    })

2023-24: answered columns per row
count    23218.0
mean       114.7
std         64.3
min          1.0
25%         49.0
50%        172.0
75%        174.0
max        174.0
Rows with 20 or fewer answered columns: 1,568 (6.8%)
2024-25: answered columns per row
count    22963.0
mean       138.5
std         60.4
min          1.0
25%         75.0
50%        179.0
75%        179.0
max        180.0
Rows with 20 or fewer answered columns: 1,323 (5.8%)


### How many rows can each objective actually use?

A row is **usable** for an objective only if it has an answer in *every* column that objective needs ("complete cases").

*Assumption to confirm with the questionnaire:* "ChatGPT user" is `Q13` in 2023-24 and `Q13a` in 2024-25.

In [6]:
letters = "abcdefghijklmno"
items_per_question = {"Q18": 12, "Q22": 10, "Q24": 7, "Q26": 10, "Q27": 10, "Q32": 15}
single_columns = ["Q10", "Q14", "Q15", "Q20", "Q33", "Q35a"]

question_columns = {}
for code_name in single_columns:
    question_columns[code_name] = [code_name]
for code_name, n_items in items_per_question.items():
    columns = []
    for letter in letters[:n_items]:
        columns.append(code_name + letter)
    question_columns[code_name] = columns

chatgpt_user_column = {"2023-24": "Q13", "2024-25": "Q13a"}     # ASSUMPTION: confirm in the questionnaire

objective_codes = {
    1: ["Q13", "Q14", "Q15", "Q18", "Q24", "Q26", "Q27", "Q35a"],
    2: ["Q15", "Q32", "Q33"],
    3: ["Q13", "Q15", "Q18", "Q20", "Q22", "Q10"],              # 'year' is added by us, so it is not in the raw files
}

for year, df in dataframes.items():
    print("=" * 70)
    print(f"{year}: rows with NO blank in the columns each objective needs")
    for number, codes in objective_codes.items():
        needed = []
        for code_name in codes:
            if code_name == "Q13":
                needed.append(chatgpt_user_column[year])
            else:
                needed.extend(question_columns[code_name])

        has_all_answers = df[needed].notna().all(axis=1)          # True only if every needed column is filled
        usable = int(has_all_answers.sum())
        pct = usable / len(df) * 100
        print(f"  Objective {number}: {usable:6,} of {len(df):,} rows ({pct:4.1f}%) using {len(needed)} columns")

        if pct < usable_alert_pct:
            severity = "High"
        elif pct < usable_warn_pct:
            severity = "Medium"
        else:
            severity = "Low"
        audit_rows.append({
            "check": f"Usable rows, Objective {number}",
            "where": f"{len(needed)} columns the objective needs",
            "wave": year,
            "finding": f"Only {usable:,} of {len(df):,} rows ({pct:.1f}%) have an answer in every column Objective {number} needs",
            "count": usable,
            "out_of": len(df),
            "severity": severity,
            "week3_idea": "Never drop incomplete rows from the cleaned file (it must stay at 15,000+ rows). Report the usable n for each analysis.",
        })

2023-24: rows with NO blank in the columns each objective needs
  Objective 1: 11,627 of 23,218 rows (50.1%) using 43 columns
  Objective 2: 11,948 of 23,218 rows (51.5%) using 17 columns
  Objective 3: 13,794 of 23,218 rows (59.4%) using 26 columns
2024-25: rows with NO blank in the columns each objective needs
  Objective 1: 14,643 of 22,963 rows (63.8%) using 43 columns
  Objective 2: 15,100 of 22,963 rows (65.8%) using 17 columns
  Objective 3: 17,026 of 22,963 rows (74.1%) using 26 columns


## 2. Duplicate rows

There is no respondent ID column, so we cannot tell a double submission from two different people who happened to give the same answers. We look at what kind of rows are duplicated to judge which is more likely.

In [7]:
for year, df in dataframes.items():
    is_duplicate = df.duplicated(keep="first")
    n_dups = int(is_duplicate.sum())
    answered_per_row = df.notna().sum(axis=1)

    dups_near_empty = int((is_duplicate & (answered_per_row <= near_empty_cutoff)).sum())
    dups_with_q15 = int((is_duplicate & df["Q15"].notna()).sum())

    # is there any column that could act as an ID? (every value filled and all different)
    id_candidates = []
    for name in df.columns:
        if df[name].notna().all() and df[name].is_unique:
            id_candidates.append(name)

    fullest = int(answered_per_row[is_duplicate].max()) if n_dups > 0 else 0

    print("=" * 70)
    print(f"{year}")
    print(f"  Exact duplicate rows            : {n_dups}")
    print(f"  ...that are near-empty rows     : {dups_near_empty}")
    print(f"  ...that answered Q15            : {dups_with_q15}")
    print(f"  Columns that could be an ID     : {id_candidates}")
    if n_dups > 0:
        print("  Answered columns in duplicated rows:")
        print(answered_per_row[is_duplicate].describe().round(1).to_string())

    audit_rows.append({
        "check": "Duplicate rows",
        "where": "whole row",
        "wave": year,
        "finding": f"{n_dups} exact duplicate rows; {dups_near_empty} are near-empty and {dups_with_q15} answered Q15; the fullest duplicated row has {fullest} answered columns; no ID column exists",
        "count": n_dups,
        "out_of": len(df),
        "severity": "Low",
        "week3_idea": "Duplicates are mostly short rows that could be different people with identical answers. Flag them (duplicated True/False) and decide whether to keep; document the reason either way.",
    })


2023-24
  Exact duplicate rows            : 30
  ...that are near-empty rows     : 29
  ...that answered Q15            : 0
  Columns that could be an ID     : []
  Answered columns in duplicated rows:
count    30.0
mean      3.8
std       8.8
min       1.0
25%       2.0
50%       2.0
75%       2.0
max      49.0


2024-25
  Exact duplicate rows            : 127
  ...that are near-empty rows     : 126
  ...that answered Q15            : 1
  Columns that could be an ID     : []
  Answered columns in duplicated rows:
count    127.0
mean       3.3
std       15.9
min        1.0
25%        1.0
50%        1.0
75%        1.0
max      179.0


## 3. Invalid numeric ranges

Two checks:

1. **Q3** looks like **age** (an assumption until the questionnaire confirms it). We test it against the provisional range in Section 0. `pd.to_numeric(..., errors="coerce")` turns text that is not a number into `NaN` so we can count it.
2. **Every other numeric column** should hold small whole numbers (1 to 5 scales, 0/1 ticks, a few category codes). We scan for negatives, fractions and anything above 5.

In [8]:
for year, df in dataframes.items():
    raw = df["Q3"]
    age = pd.to_numeric(raw, errors="coerce")               # text that is not a number becomes NaN
    text_entries = raw[age.isna() & raw.notna()]            # filled in, but not a number

    n_filled = int(raw.notna().sum())
    below = age[age < age_min]
    above = age[(age > age_max) & (age < birth_year_min)]
    birth_like = age[age >= birth_year_min]
    n_bad = len(below) + len(above) + len(birth_like)

    print("=" * 70)
    print(f"{year}: Q3 has {n_filled:,} answers (data type: {raw.dtype})")
    print(f"  Text that is not a number : {len(text_entries)} {list(text_entries.unique())}")
    print(f"  Below {age_min}                  : {len(below)}  {sorted(below.unique())}")
    print(f"  Above {age_max} (not a year)    : {len(above)}  {sorted(above.unique())}")
    print(f"  {birth_year_min} or more (like a birth year): {len(birth_like)}  {sorted(birth_like.unique())}")
    print(f"  Plausible ({age_min}-{age_max})            : {int(((age >= age_min) & (age <= age_max)).sum()):,}")

    audit_rows.append({
        "check": "Invalid range: Q3 (probably age)",
        "where": "Q3",
        "wave": year,
        "finding": f"{n_bad} of {n_filled:,} answers are outside {age_min}-{age_max} ({len(below)} too low, {len(above)} too high, {len(birth_like)} look like birth years)",
        "count": n_bad,
        "out_of": n_filled,
        "severity": "Medium",
        "week3_idea": "Flag as outliers, do not delete. Q3 is not used by our objectives; only matters if age is added to an analysis.",
    })
    audit_rows.append({
        "check": "Wrong type: Q3 holds text",
        "where": "Q3",
        "wave": year,
        "finding": f"Q3 is stored as {raw.dtype}; {len(text_entries)} entries are text, not numbers",
        "count": len(text_entries),
        "out_of": n_filled,
        "severity": "Medium" if len(text_entries) > 0 else "OK",
        "week3_idea": "Convert with pd.to_numeric(errors='coerce') after reading the numbers out of text; keep the original text.",
    })

2023-24: Q3 has 23,006 answers (data type: float64)
  Text that is not a number : 0 []
  Below 18                  : 35  [np.float64(2.0), np.float64(10.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0)]
  Above 90 (not a year)    : 12  [np.float64(93.0), np.float64(95.0), np.float64(99.0), np.float64(100.0), np.float64(101.0), np.float64(189.0), np.float64(333.0)]
  1900 or more (like a birth year): 49  [np.float64(1974.0), np.float64(1989.0), np.float64(1996.0), np.float64(1998.0), np.float64(1999.0), np.float64(2000.0), np.float64(2001.0), np.float64(2002.0), np.float64(2003.0), np.float64(2004.0), np.float64(2005.0), np.float64(2006.0)]
  Plausible (18-90)            : 22,910
2024-25: Q3 has 22,667 answers (data type: str)
  Text that is not a number : 2 ['19 سنة', '20Years']
  Below 18                  : 0  []
  Above 90 (not a year)    : 15  [np.float64(95.0), np.float64(97.0), np.float64(98.0), np.float64(99.0), np.float64(100.0)]
  1900 or more (like a 

In [9]:
for year, df in dataframes.items():
    numeric = df.select_dtypes(include="number")
    if "Q3" in numeric.columns:
        numeric = numeric.drop(columns=["Q3"])              # Q3 was checked above

    negative = []
    not_whole = []
    too_big = []
    for name in numeric.columns:
        values = numeric[name].dropna()
        if (values < 0).any():
            negative.append(name)
        if (values % 1 != 0).any():                          # % 1 is the fractional part; non-zero means not a whole number
            not_whole.append(name)
        if (values > 5).any():
            too_big.append(name)

    # how many columns share each (smallest value, biggest value) pair?
    ranges = pd.DataFrame({"min": numeric.min(), "max": numeric.max()})
    families = ranges.value_counts()

    print("=" * 70)
    print(f"{year}: {numeric.shape[1]} numeric columns scanned (Q3 excluded)")
    print(f"  Negative values   : {negative}")
    print(f"  Fractions         : {not_whole}")
    print(f"  Values above 5    : {too_big}")
    print("  Columns by (min, max) range:")
    print(families.to_string())

    n_problem = len(negative) + len(not_whole) + len(too_big)
    audit_rows.append({
        "check": "Invalid range: other numeric columns",
        "where": f"{numeric.shape[1]} numeric columns except Q3",
        "wave": year,
        "finding": ("No negative, fractional or above-5 values" if n_problem == 0
                    else f"{len(negative)} columns with negatives, {len(not_whole)} with fractions, {len(too_big)} above 5"),
        "count": n_problem,
        "out_of": numeric.shape[1],
        "severity": "OK" if n_problem == 0 else "High",
        "week3_idea": "Nothing to fix. Re-run this check against the exact valid ranges from the data dictionary.",
    })

2023-24: 169 numeric columns scanned (Q3 excluded)
  Negative values   : []
  Fractions         : []
  Values above 5    : []
  Columns by (min, max) range:
min  max
1.0  5.0    154
     3.0      7
     2.0      5
     4.0      3


2024-25: 174 numeric columns scanned (Q3 excluded)
  Negative values   : []
  Fractions         : []
  Values above 5    : []
  Columns by (min, max) range:
min  max
1.0  5.0    154
     3.0      7
0.0  1.0      6
1.0  2.0      4
     4.0      3


## 4. Inconsistent categories and text

For every **text column** we check:

* **Padded spaces:** values with a space at the start or end (`"Kenya "` is not `"Kenya"` to Python).
* **Case and spacing variants:** after making everything lower case and trimming spaces (`.str.strip().str.lower()`), do some different values become the same? That means the same category was typed in several ways.
* **Placeholder values:** entries like `n/a`, `test`, `-` or `0` that are not real answers.

In [10]:
placeholder_values = ["n/a", "na", "none", "null", "nan", "test", "xxx", "-", "--", ".", "?", "0"]

for year, df in dataframes.items():
    text_columns = list(df.select_dtypes(exclude="number").columns)
    print("=" * 70)
    print(f"{year}: text columns = {text_columns}")

    for name in text_columns:
        s = df[name].dropna().astype(str)
        normalised = s.str.strip().str.lower()
        padded = int((s != s.str.strip()).sum())
        raw_distinct = s.nunique()
        norm_distinct = normalised.nunique()
        merged = raw_distinct - norm_distinct
        n_placeholder = int(normalised.isin(placeholder_values).sum())
        print(f"  {name:10} filled={len(s):6,}  distinct={raw_distinct:6,}  after lower+strip={norm_distinct:6,}  padded={padded:3}  placeholders={n_placeholder:3}")

        if padded > 0 or merged > 0 or n_placeholder > 0:
            audit_rows.append({
                "check": f"Inconsistent text: {name}",
                "where": name,
                "wave": year,
                "finding": f"{merged} of {raw_distinct:,} distinct values are repeats once case and spaces are ignored; {padded} padded entries; {n_placeholder} placeholder-like entries",
                "count": merged,
                "out_of": raw_distinct,
                "severity": "Medium" if name == "Q13f_text" else "Low",
                "week3_idea": "Normalise with .str.strip().str.lower() into a new column; keep the original text.",
            })
        else:
            audit_rows.append({
                "check": f"Inconsistent text: {name}",
                "where": name,
                "wave": year,
                "finding": "No padded spaces, case variants or placeholder values found",
                "count": 0,
                "out_of": raw_distinct,
                "severity": "OK",
                "week3_idea": "Nothing to fix.",
            })

2023-24: text columns = ['Q1', 'Q4', 'Q5', 'source']
  Q1         filled=21,731  distinct=   149  after lower+strip=   149  padded=  0  placeholders=  0
  Q4         filled=22,917  distinct=   109  after lower+strip=   109  padded=  0  placeholders=  0
  Q5         filled=22,579  distinct= 8,725  after lower+strip= 7,072  padded=  3  placeholders=  3
  source     filled=23,218  distinct=     7  after lower+strip=     7  padded=  0  placeholders=  0
2024-25: text columns = ['Q1', 'Q3', 'Q4', 'Q5', 'Q13f_text', 'source']
  Q1         filled=21,249  distinct=   155  after lower+strip=   155  padded=  0  placeholders=  0
  Q3         filled=22,667  distinct=    64  after lower+strip=    64  padded=  0  placeholders=  0
  Q4         filled=22,551  distinct=   120  after lower+strip=   120  padded=  0  placeholders=  0
  Q5         filled=22,247  distinct= 8,662  after lower+strip= 6,964  padded=  9  placeholders=  3
  Q13f_text  filled= 1,232  distinct=   644  after lower+strip=   547  padd

### Looking closer at the free-text columns

* **Q13f_text** (2024-25 only) is where students typed the *other* chatbots they use. Typos and different spellings of the same tool would split one category into many.
* **Q5** is free text (it looks like the institution). Very short or digits-only entries are not real answers.

In [11]:
df_2425 = dataframes["2024-25"]
other_text = df_2425["Q13f_text"].dropna().astype(str).str.strip().str.lower()
print(f"Q13f_text 2024-25: {len(other_text):,} entries, {other_text.nunique():,} distinct after lower+strip")
print("Most common entries:")
print(other_text.value_counts().head(15).to_string())

# entries that are not a chatbot at all ("none" in English, Spanish, Italian)
non_answers = ["none", "ninguno", "nessuno", "no", "no uso", "nada"]
n_non_answers = int(other_text.isin(non_answers).sum())
print(f"Entries that say 'none' in some language: {n_non_answers}")

audit_rows.append({
    "check": "Free text needs mapping: Q13f_text",
    "where": "Q13f_text",
    "wave": "2024-25",
    "finding": f"{n_non_answers} entries are non-answers (none/ninguno/nessuno...) and the same tool appears under several spellings (e.g. 'notebook lm' and 'notebooklm'), so lower-casing alone is not enough",
    "count": n_non_answers,
    "out_of": len(other_text),
    "severity": "Medium",
    "week3_idea": "Only research question 7 needs this. Build a small mapping dictionary for the most common tools and put the rest in 'other'.",
})

print()
for year, df in dataframes.items():
    q5 = df["Q5"].dropna().astype(str).str.strip()
    very_short = q5[q5.str.len() <= 2]
    digits_only = q5[q5.str.fullmatch(r"\d+")]
    print(f"{year} Q5: {len(q5):,} entries | 2 characters or fewer: {len(very_short)} | digits only: {len(digits_only)}")
    print("   examples (<=2 chars):", list(very_short.unique()[:12]))

    n_junk = len(very_short) + len(digits_only)
    audit_rows.append({
        "check": "Junk entries in free text Q5",
        "where": "Q5",
        "wave": year,
        "finding": f"{len(very_short)} entries of 2 characters or fewer and {len(digits_only)} digits-only entries",
        "count": n_junk,
        "out_of": len(q5),
        "severity": "Low",
        "week3_idea": "Q5 is not used by our objectives. Flag junk entries; leave the column as is.",
    })

Q13f_text 2024-25: 1,232 entries, 547 distinct after lower+strip
Most common entries:
Q13f_text
deepseek          147
meta ai            50
poe                46
ninguno            25
kimi               20
meta               20
github copilot     19
bggpt              18
deepl              18
nessuno            18
notebook lm        17
notebooklm         16
gemini             13
none               13
blackbox           12
Entries that say 'none' in some language: 62

2023-24 Q5: 22,579 entries | 2 characters or fewer: 37 | digits only: 6
   examples (<=2 chars): ['A', 'Po', 'a', '3', '.', 'Hh', 'Uy', 'UA', 'TU', 'ТУ', '大三', 'no']


2024-25 Q5: 22,247 entries | 2 characters or fewer: 57 | digits only: 1
   examples (<=2 chars): ['No', '.', 'FU', 'TU', 'Т', '1', 'Cs', '嘉应', 'un', 'IU', 'Kh', 'C']


### Do the categories match between the two years?

Country columns (**Q1** and **Q4**) and the language column (**source**) should use the same spellings in both years. Names that appear in only one year are either different countries (a *coverage* difference) or the same country spelled differently (a *consistency* problem). We list them to tell which.

In [12]:
df_a = dataframes["2023-24"]
df_b = dataframes["2024-25"]

for name in ["Q1", "Q4"]:
    set_a = set(df_a[name].dropna())
    set_b = set(df_b[name].dropna())
    only_a = sorted(set_a - set_b)
    only_b = sorted(set_b - set_a)
    print("=" * 70)
    print(f"{name}: {len(set_a)} distinct in 2023-24, {len(set_b)} in 2024-25, {len(set_a & set_b)} in both")
    print(f"  Only in 2023-24 ({len(only_a)}): {only_a}")
    print(f"  Only in 2024-25 ({len(only_b)}): {only_b}")

    audit_rows.append({
        "check": f"Category coverage differs: {name}",
        "where": name,
        "wave": "both",
        "finding": f"{len(only_a)} names only in 2023-24 and {len(only_b)} only in 2024-25 (listed above; they read as different countries, not spelling variants)",
        "count": len(only_a) + len(only_b),
        "out_of": len(set_a | set_b),
        "severity": "Medium",
        "week3_idea": "No fix needed, but mention in limitations: the two years do not cover the same countries, so year differences may reflect who answered.",
    })

print()
print("Language column 'source' (% of rows):")
shares = pd.DataFrame({
    year: (dataframes[year]["source"].value_counts(normalize=True) * 100).round(1)
    for year in survey_years
})
print(shares.to_string())
same_languages = set(df_a["source"].dropna()) == set(df_b["source"].dropna())
print("Same set of languages in both years:", same_languages)

audit_rows.append({
    "check": "Category match: language column 'source'",
    "where": "source",
    "wave": "both",
    "finding": ("Same seven language codes in both years, but their shares differ" if same_languages
                else "The language codes differ between the years"),
    "count": len(set(df_a["source"].dropna()) ^ set(df_b["source"].dropna())),
    "out_of": len(set(df_a["source"].dropna()) | set(df_b["source"].dropna())),
    "severity": "Low" if same_languages else "Medium",
    "week3_idea": "No fix needed. The shift in language mix is another reason year differences may not be real change.",
})

Q1: 149 distinct in 2023-24, 155 in 2024-25, 135 in both
  Only in 2023-24 (14): ['Benin', 'Bolivia', 'Burundi', 'Comoros', 'Denmark', 'Equatorial Guinea', 'Gabon', 'Guinea', 'Malta', 'Mongolia', 'Niger', 'St. Vincent & The Grenadines', 'Uruguay', 'Vatican City (Holy See)']
  Only in 2024-25 (20): ['Chad', 'Dominica', 'East Timor', 'Eritrea', 'Fiji', 'Grenada', 'Haiti', 'Iceland', 'Jamaica', 'Kiribati', 'Lesotho', 'Mali', 'Namibia', 'New Zealand', 'Paraguay', 'Qatar', 'Solomon Islands', 'Swaziland', 'Uganda', 'Vanuatu']
Q4: 109 distinct in 2023-24, 120 in 2024-25, 87 in both
  Only in 2023-24 (22): ['Azerbaijan', 'Bolivia', 'Botswana', 'Central African Republic', 'Chad', 'Comoros', 'Costa Rica', 'Cote d’Ivoire', 'Denmark', 'Dominica', 'Ethiopia', 'Guinea', 'Iceland', 'Jamaica', 'Kenya', 'Kosovo', 'Malta', 'Norway', 'Sweden', 'Thailand', 'Uzbekistan', 'Vatican City (Holy See)']
  Only in 2024-25 (33): ['Cameroon', 'Cuba', 'Djibouti', 'Dominican Republic', 'East Timor', 'El Salvador', 'E

## 5. Data types and structure across the two years

If a column has a different **data type** in the two files (numbers in one, text in the other) the two tables cannot be stacked cleanly. We also list the columns that exist in only one file. (This also reconfirms the Q13 layout difference found earlier.)

In [13]:
cols_a = set(df_a.columns)
cols_b = set(df_b.columns)
shared = sorted(cols_a & cols_b)

type_mismatch = []
for name in shared:
    if str(df_a[name].dtype) != str(df_b[name].dtype):
        type_mismatch.append((name, str(df_a[name].dtype), str(df_b[name].dtype)))

only_a = sorted(cols_a - cols_b)
only_b = sorted(cols_b - cols_a)

print(f"Columns in both files: {len(shared)}")
print(f"Shared columns whose data type differs: {type_mismatch}")
print(f"Only in 2023-24: {only_a}")
print(f"Only in 2024-25: {only_b}")

print()
print("Q13 coding:")
print("  2023-24 Q13 value counts:", df_a["Q13"].value_counts(dropna=False).sort_index().to_dict())
print("  2024-25 Q13a value counts:", df_b["Q13a"].value_counts(dropna=False).sort_index().to_dict())

audit_rows.append({
    "check": "Data type differs between years",
    "where": "shared columns",
    "wave": "both",
    "finding": f"{len(type_mismatch)} of {len(shared)} shared columns have a different data type: {[m[0] for m in type_mismatch]}",
    "count": len(type_mismatch),
    "out_of": len(shared),
    "severity": "Medium" if type_mismatch else "OK",
    "week3_idea": "Convert Q3 to numbers in both years before stacking.",
})
audit_rows.append({
    "check": "Different column layout: Q13",
    "where": "Q13 vs Q13a-Q13f + Q13f_text",
    "wave": "both",
    "finding": f"{len(only_a)} column only in 2023-24 ({only_a}) and {len(only_b)} only in 2024-25 ({only_b}); coding differs (1/2 versus 1/0)",
    "count": len(only_a) + len(only_b),
    "out_of": len(cols_a | cols_b),
    "severity": "High",
    "week3_idea": "Keep the originals. Add a new chatgpt_user column built separately for each year (after the questionnaire confirms which Q13 option is ChatGPT). Per-chatbot columns exist for 2024-25 only.",
})

Columns in both files: 173
Shared columns whose data type differs: [('Q3', 'float64', 'str')]
Only in 2023-24: ['Q13']
Only in 2024-25: ['Q13a', 'Q13b', 'Q13c', 'Q13d', 'Q13e', 'Q13f', 'Q13f_text']

Q13 coding:
  2023-24 Q13 value counts: {1.0: 16010, 2.0: 6404, nan: 804}
  2024-25 Q13a value counts: {0.0: 1919, 1.0: 19693, nan: 1351}


## 6. Date parsing

The brief asks us to check that dates parse correctly. First we need to know whether the files contain any dates at all. We look for (1) column names that mention dates or times, (2) columns Pandas already reads as dates, and (3) text values that look like dates such as `12/03/2024`.

In [14]:
date_words = ["date", "time", "day", "month", "year", "stamp"]
date_pattern = r"\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}"            # e.g. 12/03/2024 or 2024-03-12

for year, df in dataframes.items():
    named_like_dates = []
    for name in df.columns:
        for word in date_words:
            if word in name.lower():
                named_like_dates.append(name)
                break

    datetime_columns = list(df.select_dtypes(include="datetime").columns)

    looks_like_dates = {}
    for name in df.select_dtypes(exclude="number").columns:
        s = df[name].dropna().astype(str)
        looks_like_dates[name] = int(s.str.contains(date_pattern, regex=True).sum())

    print("=" * 70)
    print(f"{year}")
    print(f"  Column names mentioning dates/times : {named_like_dates}")
    print(f"  Columns read as dates by Pandas     : {datetime_columns}")
    print(f"  Text values that look like dates    : {looks_like_dates}")

    found_nothing = (len(named_like_dates) == 0 and len(datetime_columns) == 0)
    audit_rows.append({
        "check": "Date parsing / time variable",
        "where": "all columns",
        "wave": year,
        "finding": ("No date or time column exists. The only time information is the survey year, which we add ourselves."
                    if found_nothing else f"Date-like columns found: {named_like_dates + datetime_columns}"),
        "count": 0 if found_nothing else len(named_like_dates) + len(datetime_columns),
        "out_of": df.shape[1],
        "severity": "High" if found_nothing else "OK",
        "week3_idea": "Nothing to parse. Ask the lecturer how to meet the Week 4 time-series line chart requirement (the only time variable has two values).",
    })

2023-24
  Column names mentioning dates/times : []
  Columns read as dates by Pandas     : []
  Text values that look like dates    : {'Q1': 0, 'Q4': 0, 'Q5': 0, 'source': 0}


2024-25
  Column names mentioning dates/times : []
  Columns read as dates by Pandas     : []
  Text values that look like dates    : {'Q1': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q13f_text': 0, 'source': 0}


## 7. The quality audit table

All the rows added above, sorted from most to least serious. `percent` is `count` divided by `out_of` (what `out_of` counts is stated in each finding). The table is also saved to `../docs/quality_audit_table.csv` so it can be used in the README and report.

In [15]:
import os

audit = pd.DataFrame(audit_rows)
audit["percent"] = (audit["count"] / audit["out_of"] * 100).round(1)

severity_order = {"High": 0, "Medium": 1, "Low": 2, "OK": 3}
audit["order"] = audit["severity"].map(severity_order)
audit = audit.sort_values(["order", "check", "wave"]).drop(columns="order").reset_index(drop=True)

compact = audit[["severity", "check", "wave", "count", "out_of", "percent"]]
print(compact.to_string())

os.makedirs("../docs", exist_ok=True)
audit.to_csv("../docs/quality_audit_table.csv", index=False)
print()
print(f"{len(audit)} audit rows saved to ../docs/quality_audit_table.csv")

   severity                                                 check     wave  count  out_of  percent
0      High            Blank values: drop-off among ChatGPT users  2023-24   3777   15927     23.7
1      High            Blank values: drop-off among ChatGPT users  2024-25   4196   19588     21.4
2      High                          Date parsing / time variable  2023-24      0     174      0.0
3      High                          Date parsing / time variable  2024-25      0     180      0.0
4      High                          Different column layout: Q13     both      8     181      4.4
5      High                              Usable rows, Objective 1  2023-24  11627   23218     50.1
6      High                              Usable rows, Objective 2  2023-24  11948   23218     51.5
7      High                              Usable rows, Objective 3  2023-24  13794   23218     59.4
8    Medium                              Blank values, whole file  2023-24    160     174     92.0
9    Mediu

In [16]:
# Findings and next steps in full, one row at a time
for position, row in audit.iterrows():
    print(f"[{row['severity']}] {row['check']} ({row['wave']})")
    print(f"    Finding    : {row['finding']}")
    print(f"    Evidence   : {row['count']:,} of {row['out_of']:,} ({row['percent']}%)")
    print(f"    Week 3 idea: {row['week3_idea']}")
    print()

[High] Blank values: drop-off among ChatGPT users (2023-24)
    Finding    : Among rows that answered Q15, 0.2%-23.7% are blank on the key objective questions (highest: Q33)
    Evidence   : 3,777 of 15,927 (23.7%)
    Week 3 idea: Do not drop these rows from the cleaned file. For each analysis keep only rows with answers in the columns it uses, and report that n.

[High] Blank values: drop-off among ChatGPT users (2024-25)
    Finding    : Among rows that answered Q15, 0.4%-21.4% are blank on the key objective questions (highest: Q33)
    Evidence   : 4,196 of 19,588 (21.4%)
    Week 3 idea: Do not drop these rows from the cleaned file. For each analysis keep only rows with answers in the columns it uses, and report that n.

[High] Date parsing / time variable (2023-24)
    Finding    : No date or time column exists. The only time information is the survey year, which we add ourselves.
    Evidence   : 0 of 174 (0.0%)
    Week 3 idea: Nothing to parse. Ask the lecturer how to meet the

In [17]:
# How many audit rows of each severity, per survey year
print(pd.crosstab(audit["severity"], audit["wave"]).to_string())

wave      2023-24  2024-25  both
severity                        
High            5        2     1
Low             5        5     1
Medium          2        8     3
OK              5        5     0


### Headline results (read from the outputs above, using the provisional thresholds in Section 0)

* **Drop-off among ChatGPT users:** of the rows that answered Q15, 23.7% (2023-24) and 21.4% (2024-25) are blank on Q33, and 23.6% and 21.4% on Q35a. Q32a is blank for 21.8% and 19.7%.
* **Usable rows are far fewer than the headline row counts.** Rows with an answer in every column an objective needs: Objective 1 has 11,627 (2023-24) and 14,643 (2024-25); Objective 2 has 11,948 and 15,100; Objective 3 has 13,794 and 17,026.
* **Q3 (probably age):** 96 of 23,006 answers (2023-24) and 17 of 22,667 (2024-25) are outside 18-90, many of them birth years. In 2024-25, Q3 is text because of two entries (`'19 سنة'` and `'20Years'`).
* **Duplicates:** 30 (2023-24) and 127 (2024-25). Almost all are near-empty rows with one or two answers, so they are probably different people with identical short answers, not double submissions.
* **Q13 is laid out differently** in the two years (1 column versus 6 plus a text column) and coded differently (1/2 versus 1/0).
* **No date or time column** exists in either file.
* **The years cover different countries and languages** (for example 55 country names in Q4 appear in only one year), so year differences may reflect who answered.
* **Clean checks:** no negative, fractional or above-5 values in the other numeric columns, no padded spaces or case variants in the country and language columns, and the same seven language codes in both years.

## 8. How this audit feeds the rest of Week 2

* **For the observations and data-quality issues (Week 2, item 6):** pick findings from the *High* and *Medium* rows of the table and quote the evidence column.
* **For the data dictionary:** the "expected missing" notes and the Q3 and Q13 findings belong in it.
* **For Week 3 cleaning:** the last column of the table lists a treatment idea for each finding. Every cleaning decision still needs a documented reason in `docs/cleaning_log.md`.

**Limits of this audit**

* Valid ranges and the meaning of codes are provisional until the questionnaires are checked (especially Q3, Q13, Q14 and Q20).
* "ChatGPT user" (Q13 in 2023-24, Q13a in 2024-25) is an assumption.
* The duplicate check cannot separate double submissions from different people with the same answers, because the files have no respondent ID.
* Each year is a separate convenience sample, so differences between years cannot be read as change in the same students.